# Adult Income — End-to-End Data Science Project

## Project Overview
This project implements a complete Data Science pipeline in Python using the UCI Adult (Census Income) dataset.

### Objectives
- Acquire and inspect a public dataset.
- Clean and preprocess numerical and categorical data.
- Perform exploratory data analysis and visualization.
- Apply unsupervised learning with K-Means clustering.
- Build a supervised Logistic Regression classifier.
- Build a TensorFlow/Keras Deep Neural Network.
- Compare model performance using appropriate evaluation metrics.
- Derive insights, limitations, and recommendations.

**Problem:** Predict whether annual income is `<=50K` or `>50K`, while also discovering descriptive groups in the population through clustering.


In [ ]:
# Install/import required packages
import sys, subprocess, importlib.util

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])

for pkg, imp in [
    ("ucimlrepo", "ucimlrepo"),
    ("scikit-learn", "sklearn"),
    ("tensorflow", "tensorflow")
]:
    ensure_package(pkg, imp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    silhouette_score, ConfusionMatrixDisplay, RocCurveDisplay
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Libraries imported successfully.")
print("TensorFlow:", tf.__version__)


## 1. Data Acquisition

In [ ]:
# Acquire the UCI Adult dataset
adult = fetch_ucirepo(id=2)

X_raw = adult.data.features.copy()
y_raw = adult.data.targets.copy()

X_raw.columns = X_raw.columns.astype(str).str.strip()
y_raw = y_raw.iloc[:, 0].astype(str).str.strip().str.replace(".", "", regex=False)

print("Dataset shape:", X_raw.shape)
print("Number of predictor features:", X_raw.shape[1])
print("\nTarget distribution:")
print(y_raw.value_counts())


## 2. Data Inspection and Cleaning

The dataset contains categorical and numerical variables. The `?` marker is treated as missing data. The target labels are normalized so that both dotted and non-dotted versions map to the same two classes.


In [ ]:
# Inspect data types and missing markers
print("Data types:")
print(X_raw.dtypes)

marker_counts = {
    col: X_raw[col].astype(str).str.strip().eq("?").sum()
    for col in X_raw.columns
}
print("\nMissing-value markers:")
display(pd.Series(marker_counts).sort_values(ascending=False))

X = X_raw.replace(r"^\s*\?$", np.nan, regex=True)

print("\nTotal missing cells after marker conversion:",
      int(X.isna().sum().sum()))


In [ ]:
# Remove duplicate rows for the cleaned exploratory dataset
before_rows = len(X)
duplicate_count = int(X.duplicated().sum())
X_clean = X.drop_duplicates().copy()

print("Duplicate rows detected:", duplicate_count)
print("Rows before:", before_rows)
print("Rows after:", len(X_clean))
print("Remaining missing cells:", int(X_clean.isna().sum().sum()))


## 3. Exploratory Data Analysis

EDA is performed before modeling to understand distributions, class imbalance, categorical frequencies, and relationships among numerical variables.


In [ ]:
# Numeric descriptive statistics
numeric_cols = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]
display(X_clean[numeric_cols].describe().T)


In [ ]:
# Income distribution
income_counts = y_raw.value_counts()
print(income_counts)

plt.figure(figsize=(7, 4.5))
income_counts.plot(kind="bar")
plt.title("Income Class Distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Selected categorical distributions
figures = [
    ("workclass", "Workclass Distribution"),
    ("education", "Education Distribution"),
    ("marital-status", "Marital Status Distribution"),
    ("occupation", "Occupation Distribution"),
    ("race", "Race Distribution")
]

for col, title in figures:
    plt.figure(figsize=(9, 4.5))
    X_clean[col].fillna("Missing").value_counts().head(12).plot(kind="bar")
    plt.title(title)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Income by sex
cross = pd.crosstab(
    X_clean["sex"].fillna("Missing"),
    y_raw.loc[X_clean.index]
)
print(cross)

cross.plot(kind="bar", figsize=(7, 4.5))
plt.title("Income Distribution by Sex")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap for numerical variables
plt.figure(figsize=(8, 6))
sns.heatmap(X_clean[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numerical Feature Correlation")
plt.tight_layout()
plt.show()


### EDA Interpretation
The target is imbalanced toward the `<=50K` class. Most observations are from the United States, White race, Private workclass, and common education categories such as HS-grad and Some-college. Numerical correlations are generally weak to moderate, suggesting that income prediction depends on a combination of variables rather than a single strongly correlated numerical feature.


## 4. Modeling Dataset Preparation

For predictive modeling, preprocessing is fitted on the training data to reduce leakage. Numerical features are median-imputed and standardized. Categorical features are most-frequent imputed and one-hot encoded. `capital-gain` and `capital-loss` are log-transformed because of their strong right skew.


In [ ]:
# Prepare modeling data
X_model = X_raw.replace(r"^\s*\?$", np.nan, regex=True).copy()

numeric_features = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]
categorical_features = [
    "workclass", "education", "marital-status",
    "occupation", "relationship", "race", "sex", "native-country"
]

for col in numeric_features:
    X_model[col] = pd.to_numeric(X_model[col], errors="coerce")

X_model["capital-gain"] = np.log1p(X_model["capital-gain"])
X_model["capital-loss"] = np.log1p(X_model["capital-loss"])

y_binary = (y_raw == ">50K").astype(int)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_model, y_binary, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y_binary
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_train_val
)

try:
    ohe_dense = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_dense = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe_dense)
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_p = np.asarray(preprocessor.fit_transform(X_train), dtype=np.float32)
X_val_p = np.asarray(preprocessor.transform(X_val), dtype=np.float32)
X_test_p = np.asarray(preprocessor.transform(X_test), dtype=np.float32)

print("Train:", X_train_p.shape)
print("Validation:", X_val_p.shape)
print("Test:", X_test_p.shape)
print("NaNs:", np.isnan(X_train_p).sum(),
      np.isnan(X_val_p).sum(), np.isnan(X_test_p).sum())


## 5. Unsupervised Learning — K-Means Clustering

K-Means is used to identify descriptive groups in the processed feature space. Several values of K are evaluated using inertia and silhouette score. The clustering is exploratory; cluster labels are not treated as causal or as ground-truth classes.


In [ ]:
# Evaluate K values using a fixed sample for efficiency
sample_n = min(5000, len(X_train_p))
sample = X_train_p[:sample_n]

k_results = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(sample)
    sil = silhouette_score(sample, labels)
    k_results.append((k, km.inertia_, sil))
    print(f"K={k}: inertia={km.inertia_:.4f}, silhouette={sil:.4f}")

k_results_df = pd.DataFrame(
    k_results, columns=["K", "Inertia", "Silhouette"]
)
display(k_results_df)

best_k = int(k_results_df.loc[k_results_df["Silhouette"].idxmax(), "K"])
print("Selected K:", best_k)


In [ ]:
# Fit final K-Means on all processed modeling records
X_all_p = np.vstack([X_train_p, X_val_p, X_test_p])

final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)
cluster_labels = final_kmeans.fit_predict(X_all_p)

print("Final clustering inertia:", round(final_kmeans.inertia_, 2))
print("\nCluster sizes:")
print(pd.Series(cluster_labels).value_counts().sort_index())


In [ ]:
# PCA visualization of clusters
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_all_p)

print("PCA explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:",
      round(pca.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(8, 6))
plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=cluster_labels, s=8, alpha=0.25
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters in PCA Space")
plt.tight_layout()
plt.show()


## 6. Supervised Learning — Logistic Regression

Logistic Regression provides an interpretable classical baseline for the binary income classification problem.


In [ ]:
# Logistic Regression baseline
log_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    solver="liblinear",
    random_state=RANDOM_STATE
)
log_model.fit(X_train_p, y_train)

log_prob = log_model.predict_proba(X_test_p)[:, 1]
log_pred = (log_prob >= 0.5).astype(int)

log_metrics = {
    "Accuracy": accuracy_score(y_test, log_pred),
    "Precision": precision_score(y_test, log_pred),
    "Recall": recall_score(y_test, log_pred),
    "F1-score": f1_score(y_test, log_pred),
    "ROC-AUC": roc_auc_score(y_test, log_prob)
}

print("Logistic Regression test performance:")
for name, value in log_metrics.items():
    print(f"{name}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, log_pred,
    target_names=["<=50K", ">50K"], digits=4
))


## 7. Deep Learning — TensorFlow/Keras DNN

A fully connected neural network is used to learn nonlinear patterns from the processed tabular features.


In [ ]:
# Build DNN
input_dim = X_train_p.shape[1]

dnn = keras.Sequential([
    keras.Input(shape=(input_dim,), name="input"),
    layers.Dense(128, activation="relu", name="dense_128"),
    layers.Dropout(0.30, name="dropout_30"),
    layers.Dense(64, activation="relu", name="dense_64"),
    layers.Dropout(0.20, name="dropout_20"),
    layers.Dense(1, activation="sigmoid", name="output")
], name="Adult_Income_DNN")

dnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

dnn.summary()


In [ ]:
# Train DNN with overfitting controls
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

history = dnn.fit(
    X_train_p, y_train,
    validation_data=(X_val_p, y_val),
    epochs=50,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("Epochs trained:", len(history.history["loss"]))


In [ ]:
# DNN learning curves
plt.figure(figsize=(8, 4.5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("DNN Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("DNN Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# DNN test evaluation
dnn_test = dnn.evaluate(X_test_p, y_test, verbose=0)
dnn_prob = dnn.predict(X_test_p, batch_size=1024, verbose=0).ravel()
dnn_pred = (dnn_prob >= 0.5).astype(int)

dnn_metrics = {
    "Accuracy": accuracy_score(y_test, dnn_pred),
    "Precision": precision_score(y_test, dnn_pred),
    "Recall": recall_score(y_test, dnn_pred),
    "F1-score": f1_score(y_test, dnn_pred),
    "ROC-AUC": roc_auc_score(y_test, dnn_prob)
}

print("DNN test performance:")
for name, value in dnn_metrics.items():
    print(f"{name}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, dnn_pred,
    target_names=["<=50K", ">50K"], digits=4
))


In [ ]:
# Confusion matrices and ROC curves
print("DNN confusion matrix:")
print(confusion_matrix(y_test, dnn_pred))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, dnn_pred),
    display_labels=["<=50K", ">50K"]
).plot()
plt.title("DNN Confusion Matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(
    y_test, dnn_prob, name=f"DNN (AUC={dnn_metrics['ROC-AUC']:.4f})"
)
plt.title("DNN ROC Curve")
plt.tight_layout()
plt.show()


## 8. Model Comparison and Evaluation

The models address the same binary prediction problem. Accuracy, precision, recall, F1-score, and ROC-AUC are considered together because the target classes are imbalanced.

K-Means is evaluated separately using inertia and silhouette score because it is an unsupervised descriptive method rather than a classifier.


In [ ]:
# Compare supervised models
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Logistic Regression": [log_metrics[m] for m in
                            ["Accuracy","Precision","Recall","F1-score","ROC-AUC"]],
    "DNN": [dnn_metrics[m] for m in
            ["Accuracy","Precision","Recall","F1-score","ROC-AUC"]]
})

display(comparison)

comparison.set_index("Metric").plot(kind="bar", figsize=(9, 5))
plt.title("Supervised Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Insights and Recommendations

### Key insights
- The income target is imbalanced, with `<=50K` forming the majority class.
- Education, occupation, marital status, relationship, age, hours worked, and financial attributes provide useful predictive information.
- K-Means identifies descriptive groups, but the low silhouette values indicate substantial overlap among clusters.
- Logistic Regression provides a strong classical baseline.
- The DNN provides strong nonlinear classification performance, especially in ROC-AUC.

### Recommendations
1. Use multiple metrics rather than accuracy alone.
2. Consider threshold tuning when the cost of missing `>50K` cases is high.
3. Compare deep learning against strong tabular models such as gradient boosting.
4. Explore class weighting or focal loss if minority-class recall is a priority.
5. Investigate cluster profiles further for descriptive segmentation.
6. Treat model outputs as statistical predictions, not causal explanations.


## 10. Conclusion

This notebook demonstrates an end-to-end Data Science workflow: public data acquisition, cleaning, preprocessing, EDA, unsupervised clustering, supervised classification, deep learning, evaluation, interpretation, and recommendations.

The project is designed as a standalone portfolio artifact and does not depend on any particular course or internship.
